In [1]:
import random
from itertools import combinations
from collections import Counter

# ---- Card representation ----
RANKS = list(range(2, 15))  # 2-14, 14 = Ace
SUITS = ['s', 'h', 'd', 'c']
DECK = [(r, s) for r in RANKS for s in SUITS]

# ---- Hand evaluation (identical to main file) ----

def best_hand_rank(cards):
    best = None
    for combo in combinations(cards, min(5, len(cards))):
        rank = hand_rank(combo)
        if best is None or rank > best:
            best = rank
    return best

def hand_rank(cards):
    ranks = sorted([r for r, s in cards], reverse=True)
    suits = [s for r, s in cards]
    counts = Counter(ranks)
    rank_counts = sorted(counts.values(), reverse=True)

    is_flush = len(set(suits)) == 1
    is_straight = (len(set(ranks)) == 5 and ranks[0] - ranks[4] == 4)

    if set(ranks) == {14, 2, 3, 4, 5}:
        is_straight = True
        ranks = [5, 4, 3, 2, 1]
        counts = Counter(ranks)

    tiebreaker = sorted(ranks, key=lambda r: (counts[r], r), reverse=True)

    if is_straight and is_flush:
        return (8, tiebreaker)
    if rank_counts[0] == 4:
        return (7, tiebreaker)
    if rank_counts[:2] == [3, 2]:
        return (6, tiebreaker)
    if is_flush:
        return (5, tiebreaker)
    if is_straight:
        return (4, tiebreaker)
    if rank_counts[0] == 3:
        return (3, tiebreaker)
    if rank_counts[:2] == [2, 2]:
        return (2, tiebreaker)
    if rank_counts[0] == 2:
        return (1, tiebreaker)
    return (0, tiebreaker)

# ---- Full rule set ----

def generate_rules():
    rules = []
    rules.append(('any', lambda c: True))
    for suit in SUITS:
        rules.append((f'suit_{suit}', lambda c, s=suit: c[1] == s))
    for rank in RANKS:
        rules.append((f'rank_gte_{rank}', lambda c, r=rank: c[0] >= r))
    for rank in RANKS:
        rules.append((f'rank_eq_{rank}', lambda c, r=rank: c[0] == r))
    for suit in SUITS:
        for rank in RANKS:
            rules.append((f'suit_{suit}_rank_gte_{rank}',
                          lambda c, s=suit, r=rank: c[1] == s and c[0] >= r))
    return rules

ALL_RULES = generate_rules()

# ---- Single game simulation ----

def play_game_random(n_picks=5, dealer_fill=8):
    """Play one full game using a random rule each turn."""
    my_hand = []
    dealer_hand = []
    remaining_deck = list(DECK)

    for _ in range(n_picks):
        if not remaining_deck:
            break

        # Pick a random rule that matches at least one card in the deck
        valid_rules = [(name, fn) for name, fn in ALL_RULES
                       if any(fn(c) for c in remaining_deck)]
        if not valid_rules:
            break
        rule_name, rule_fn = random.choice(valid_rules)

        # Shuffle and find first matching card
        random.shuffle(remaining_deck)
        target_idx = next(i for i, c in enumerate(remaining_deck) if rule_fn(c))

        my_hand.append(remaining_deck[target_idx])
        dealer_hand += remaining_deck[:target_idx]
        remaining_deck = remaining_deck[target_idx + 1:]

    if len(my_hand) < 5:
        return 0  # incomplete hand loses

    # Dealer fills to dealer_fill cards
    random.shuffle(remaining_deck)
    needed = max(0, dealer_fill - len(dealer_hand))
    final_dealer = dealer_hand + remaining_deck[:needed]

    return 1 if best_hand_rank(my_hand) > best_hand_rank(final_dealer) else 0

# ---- Run simulation ----

def simulate_random(n_games=2000, n_picks=5, dealer_fill=8):
    wins = sum(play_game_random(n_picks, dealer_fill) for _ in range(n_games))
    win_rate = wins / n_games
    return win_rate

if __name__ == '__main__':
    print("Simulating random rule strategy...")
    n_games = 2000
    win_rate = simulate_random(n_games=n_games)
    print(f"Random rule strategy win rate over {n_games} games: {win_rate:.3f}")


Simulating random rule strategy...
Random rule strategy win rate over 2000 games: 0.014


In [2]:
def play_game_random_debug(n_picks=5, dealer_fill=8):
    my_hand = []
    dealer_hand = []
    remaining_deck = list(DECK)

    for _ in range(n_picks):
        if not remaining_deck:
            break
        valid_rules = [(name, fn) for name, fn in ALL_RULES
                       if any(fn(c) for c in remaining_deck)]
        rule_name, rule_fn = random.choice(valid_rules)
        random.shuffle(remaining_deck)
        target_idx = next(i for i, c in enumerate(remaining_deck) if rule_fn(c))
        print(f"  Rule: {rule_name}, cards passed to dealer: {target_idx}")
        my_hand.append(remaining_deck[target_idx])
        dealer_hand += remaining_deck[:target_idx]
        remaining_deck = remaining_deck[target_idx + 1:]

    print(f"  My hand: {my_hand}")
    print(f"  Dealer got {len(dealer_hand)} cards before fill")

play_game_random_debug()

  Rule: suit_h_rank_gte_6, cards passed to dealer: 0
  Rule: suit_s_rank_gte_6, cards passed to dealer: 3
  Rule: rank_gte_8, cards passed to dealer: 1
  Rule: rank_gte_4, cards passed to dealer: 0
  Rule: suit_h_rank_gte_9, cards passed to dealer: 1
  My hand: [(9, 'h'), (10, 's'), (13, 'd'), (8, 's'), (12, 'h')]
  Dealer got 5 cards before fill
